In [2]:
%load_ext autoreload
%autoreload 2

import analysis_tools.data_loading as dl
import analysis_tools.plotting as plt
import analysis_tools.signal_analysis as sa
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# Load data
from pathlib import Path


# dir_path = Path("/home/agilicious/catkin_ws/ros_logs/recording_2025-08-18_16-47-38")
# dir_path = Path("/home/agilicious/catkin_ws/ros_logs/recording_2025-08-18_19-45-37")
# dir_path = Path("/home/agilicious/catkin_ws/ros_logs/recording_2025-08-18_19-47-02")
# dir_path = Path("/home/agilicious/catkin_ws/ros_logs/sim2sim_1")
dir_path = Path("/home/agilicious/catkin_ws/ros_logs/hil_side_vel")
command_path = dir_path / "feed_command.csv"
estimates_path = dir_path / "quad_state.csv"

commands = dl.Command.from_csv(command_path)
state_estimates = dl.QuadStateEstimates.from_csv(estimates_path)

No timestamp violations found in Command.
No timestamp violations found in QuadStateEstimates.


In [11]:
fig = go.Figure()
plt.add_3d_trajectory_traces(
    fig, state_estimates.time.to_numpy(), state_estimates.position.to_numpy(), label_prefix="EKF", line=dict(color='blue', width=2)
)

fig.update_layout(
    title="EKF vs Vicon Trajectory",
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode='data'
    ),
    legend=dict(
        title="Legend",
        x=0.01,
        y=0.99,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='black',
        borderwidth=1
    )
)
fig.show()  

In [12]:
# Plot pos Wx,y,z components
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Smoothed Vicon X", "Smoothed Vicon Y", "Smoothed Vicon Z"),
    vertical_spacing=0.1
)

t_min = state_estimates.time.to_numpy()[0]
t = state_estimates.time.to_numpy() - t_min
print(t_min)
plt.add_xyz_traces_stacked(fig, t, state_estimates.velocity_angular.to_numpy(), label_prefix="EKF", line=dict(color='blue'))


fig.update_layout(
    title="Smoothed Vicon vs EKF Angular Velocity Components",
    xaxis_title="Time (s)",
    yaxis_title="Velocity (rad/s)",
    legend=dict(
        title="Legend",
        x=0.01,
        y=0.99,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='black',
        borderwidth=1
    )
)
fig.show()

1755709082.262318


In [13]:
# Plot pos Wx,y,z components
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Smoothed Vicon X", "Smoothed Vicon Y", "Smoothed Vicon Z"),
    vertical_spacing=0.1
)
t_min = min(state_estimates.time.to_numpy()[0], commands.time.to_numpy()[0])
print(f"t_min: {t_min}")
t_state = state_estimates.time.to_numpy() - t_min
t_cmd = commands.time.to_numpy() - t_min
plt.add_xyz_traces_stacked(fig, t_state, state_estimates.velocity_angular.to_numpy(), label_prefix="Body Rate", line=dict(color='green'))
plt.add_xyz_traces_stacked(fig, t_cmd, commands.body_rates.to_numpy(), label_prefix="Body Rate Cmd", line=dict(color='blue'))
fig.show()


t_min: 1755709082.262318


In [ ]:
# --- Analyze delay between commands and state estimates body rates ---
# Get a specific time interval for analysis
# time_interval = [0.8, 2.4]
time_interval = [3, 15]


t_state = state_estimates.time.to_numpy() - t_min
t_cmd = commands.time.to_numpy() - t_min

state_interval_idx = (t_state >= time_interval[0]) & (t_state <= time_interval[1])
cmd_interval_idx = (t_cmd >= time_interval[0]) & (t_cmd <= time_interval[1])

state_body_rates = state_estimates.velocity_angular.to_numpy()[state_interval_idx]
t_states = t_state[state_interval_idx]
cmd_body_rates = commands.body_rates.to_numpy()[cmd_interval_idx]
t_cmds = t_cmd[cmd_interval_idx]


i = 0
lag_samples, delay_x, cmd_x_aligned = sa.run_delay_analysis_nonuniform(state_body_rates[:,i], t_states, cmd_body_rates[:,i], t_cmds)
i = 1
lag_samples, delay_y, cmd_y_aligned = sa.run_delay_analysis_nonuniform(state_body_rates[:,i], t_states, cmd_body_rates[:,i], t_cmds)
i = 2
lag_samples, delay_z, cmd_z_aligned = sa.run_delay_analysis_nonuniform(state_body_rates[:,i], t_states, cmd_body_rates[:,i], t_cmds)

print(f"Delay X: {delay_x:.5f} s, Y: {delay_y:.5f} s, Z: {delay_z:.5f} s")
print(f"Avg Delay: {(delay_x + delay_y + delay_z) / 3:.5f} s")

cmd_body_rate_aligned = np.vstack((cmd_x_aligned, cmd_y_aligned, cmd_z_aligned)).T

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Aligned Body Rate", "Delay Analysis Correlation"),
    vertical_spacing=0.1
)
plt.add_xyz_traces_stacked(fig, t_cmds, cmd_body_rate_aligned, label_prefix="Commanded Body Rate", line=dict(color='blue'))
plt.add_xyz_traces_stacked(fig, t_states, state_body_rates, label_prefix="Aligned Body Rate", line=dict(color='green'))

fig.show()

Delay X: -0.06760 s, Y: -0.06930 s, Z: -0.15139 s
Avg Delay: -0.09610 s
